# 03 — Local validation against manual annotations

Compares saved predictions to a small manually annotated ground-truth CSV (`ct_id` column plus one 0/1 column per label). Point `ANNOTATIONS_PATH` at that file once it exists.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
import pandas as pd

from report2label.extraction.label_mapper import LabelVocabulary
from report2label.utils.io import read_yaml
from report2label.validation.error_analysis import find_mismatches
from report2label.validation.evaluator import evaluate_predictions, load_predictions

pipeline_config = read_yaml(PROJECT_ROOT / "configs" / "pipeline.yaml")
label_vocab = LabelVocabulary.from_yaml(PROJECT_ROOT / "configs" / "labels.yaml")

PREDICTIONS_DIR = PROJECT_ROOT / pipeline_config["paths"]["predictions_dir"]
ANNOTATIONS_PATH = PROJECT_ROOT / pipeline_config["paths"]["annotations_dir"] / "manual_labels.csv"

In [ ]:
predictions = load_predictions(PREDICTIONS_DIR)
metrics = evaluate_predictions(predictions, ANNOTATIONS_PATH, label_vocab.names)

print(f"Matched {metrics['n_matched']} annotated report(s)")
print(metrics["overall"])

pd.DataFrame(metrics["per_label"]).T

In [ ]:
annotations = pd.read_csv(ANNOTATIONS_PATH, dtype={"ct_id": str})
mismatches = find_mismatches(predictions, annotations, label_vocab.names)
pd.DataFrame(mismatches)